## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy.stats import entropy
from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Parameter

In [4]:
current_working_directory = os.getcwd()
print(current_working_directory)

/Users/rapido/analytics/customer/Ads-Monetisation/captian_dispatch


In [5]:
local_datasets = '/Users/rapido/local-datasets/customer/ads-monetisation/captian_dispatch_exp/quick_funnel/raw/'

In [6]:
## Parameter 
start_date = '20240301'
end_date = '20240315'
cities = ['Bangalore', 'Jaipur', 'Hyderabad', 'Delhi']
service = ['CabEconomy', 'Bike Lite', 'Link', 'Auto']

In [7]:
# Convert the lists to comma-separated strings
city_str = "', '".join(cities)
print(city_str)

service_str = "', '".join(service)
print(service_str)

Bangalore', 'Jaipur', 'Hyderabad', 'Delhi
CabEconomy', 'Bike Lite', 'Link', 'Auto


## Datasets

### FE data 

In [9]:
user_base_query = f"""

    WITH fe_tbl AS (

        SELECT
            yyyymmdd,
            user_id as user_id,
            fare_estimate_id
        FROM
            hive.pricing.fare_estimates_enriched
        WHERE 
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND city IN ('{city_str}')
            AND service_name IN ('{service_str}')
            AND lower(platform) IN ('android')
        ),

        rr_net_tbl as (

        SELECT   
            yyyymmdd,
            city_name AS city,
            service_obj_service_name AS service_name,
            customer_id,
            estimate_id AS fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag,
            modified_order_status
        FROM orders.order_logs_fact ols
        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND city_name IN ('{city_str}')
            AND service_obj_service_name IN ('{service_str}')
            AND channel_host IN ('android')
        ),

        fe_merged AS (

        SELECT
            user_id,
            COUNT(distinct fare_estimate_id) AS fe_count,
            COUNT(distinct order_id) AS rr_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count,

            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'expired' THEN order_id END) expired,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'aborted' THEN order_id END) aborted,

            COALESCE(TRY(COUNT(distinct order_id)*100.00/COUNT(distinct fare_estimate_id)), 0) FE_RR,
            COALESCE(TRY(COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END)*100.00/COUNT(distinct order_id)), 0) G2N,
            COALESCE(TRY(COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END)*100.00/COUNT(distinct fare_estimate_id)), 0) FE_NET

        FROM        
            (
            SELECT
                fe_tbl.*,
                rr_net_tbl.customer_id,
                rr_net_tbl.order_id,
                rr_net_tbl.order_status,
                rr_net_tbl.modified_order_status,
                rr_net_tbl.spd_fraud_flag
            FROM
                fe_tbl
            LEFT JOIN
                rr_net_tbl
                ON fe_tbl.yyyymmdd = rr_net_tbl.yyyymmdd
                AND fe_tbl.fare_estimate_id = rr_net_tbl.fare_estimate_id
            )
        GROUP BY 1
        HAVING COUNT(distinct order_id) > 0
        ),
        
        


        exp_base AS (

        SELECT 
            event_props_user_id userId
        FROM 
            clevertap.customer_banner_monetisation_immutable 
        WHERE 
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND event_props_current_city IN ('Bangalore', 'Jaipur','Hyderabad', 'Delhi' )
            AND event_props_action IN ('view', 'click')
            AND event_props_type = 'GAMBanner'
            AND event_props_sub_type LIKE 'native%'
            AND event_props_screen_name = 'CaptainSearchScreen'
            AND event_props_placement IN ( 'CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd')
        GROUP BY 1
        )

        SELECT
            CASE 
            WHEN b.userId IS NULL THEN 'CONTROL'
            ELSE 'TEST' END AS exp_group,
            taxi_lifetime_stage segment,
            a.*
        FROM 
            fe_merged a
        LEFT JOIN
            datasets.iallocator_customer_segments s
            ON run_date = '2024-03-04'
            AND s.customer_id = a.user_id
            
        LEFT JOIN
            exp_base b
            ON a.user_id = b.userId            
"""

df_user_base = pd.read_sql(user_base_query, connection)
df_user_base.to_csv(local_datasets + 'df_user_base_v1.csv' , index=False)

In [10]:
df_user_base = pd.read_csv(local_datasets + 'df_user_base_v1.csv')
df_user_base

,exp_group,segment,user_id,fe_count,rr_count,net_count,cobrm,cobra,ocara,expired,aborted,FE_RR,G2N,FE_NET
0,TEST,COMMITTED,5c62be84f2edc7336756baba,17,5,4,0,1,0,0,0,29.41,80.00,23.53
1,CONTROL,HOOK,5e66ed4e94fa355604220ec6,6,1,0,0,1,0,0,0,16.67,0.00,0.00
2,CONTROL,SUSTENANCE,5b08301d2c9699373e463718,23,6,5,0,0,1,0,0,26.09,83.33,21.74
3,CONTROL,SUSTENANCE,64e2ced5fe0362730cc3c06a,12,4,3,0,0,1,0,0,33.33,75.00,25.00
4,TEST,HOOK,61448bbede864ca0fbffa996,25,14,9,0,2,0,3,0,56.00,64.29,36.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4433883,CONTROL,UNKNOWN,631d54c482761722fbbcd6a2,9,4,4,0,0,0,0,0,44.44,100.00,44.44
4433884,CONTROL,CHURN_OTB,64a77a11d1af075fed880429,4,1,1,0,0,0,0,0,25.00,100.00,25.00
4433885,CONTROL,SUSTENANCE,5c96221cb50ab0456be2b21f,6,2,2,0,0,0,0,0,33.33,100.00,33.33
4433886,CONTROL,DORMANT,62447581aee22963a610e755,3,1,0,0,0,1,0,0,33.33,0.00,0.00


In [11]:
df_user_base.groupby(['exp_group']).agg(customer = ('user_id', 'nunique'),
                                                   fe = ('fe_count', 'mean'),
                                                   rr = ('rr_count', 'mean'),
                                                  )

,customer,fe,rr
exp_group,,,
CONTROL,4039844,9.031273,3.583506
TEST,394044,20.006560,9.118870


In [12]:
# df1_user_base = df_user_base ##[df_user_base['segment'].isin(['HANDHOLDING', 'SUSTENANCE'])]
df1_user_base = df_user_base[
#     df_user_base['segment'].isin(['SUSTENANCE', 'HOOK'])
#                              &
                             df_user_base['rr_count'] < 6
                            ]

In [13]:
# df1_user_base.groupby(['exp_group', 'segment']).agg(customer = ('user_id', 'nunique'))

In [14]:
control_group = df1_user_base[df1_user_base['exp_group'] == 'CONTROL']
test_group = df1_user_base[df1_user_base['exp_group'] == 'TEST']

# Find the minimum count of data points between the two groups
min_count = min(len(control_group), len(test_group))

# Sample data points from each group with the minimum count
homogeneous_sample_control = control_group.sample(min_count, replace=False)
homogeneous_sample_test = test_group.sample(min_count-25000, replace=False)

# Concatenate the sampled groups back into one DataFrame
homogeneous_sample = pd.concat([homogeneous_sample_control, homogeneous_sample_test])

In [15]:
print(homogeneous_sample['exp_group'].value_counts())

exp_group
CONTROL    196619
TEST       171619
Name: count, dtype: int64


In [16]:
test = homogeneous_sample\
.groupby(['exp_group'])\
.agg(user_count = ('user_id', 'nunique'),
     fe_count = ('fe_count', 'sum'),
     rr_count = ('rr_count', 'sum'),
     net_count = ('net_count', 'sum'),
     cobrm = ('cobrm', 'sum'),
     cobra = ('cobra', 'sum'),
     ocara = ('ocara', 'sum')
    )\
.reset_index()

test['fe2rr'] = (test['rr_count']*100.00/test['fe_count']).round(2)
test['g2n'] = (test['net_count']*100.00/test['rr_count']).round(2)
test['fe2net'] = (test['net_count']*100.00/test['fe_count']).round(2)
test['% cobrm'] = (test['cobrm']*100.00/test['rr_count']).round(2)
test['% cobra'] = (test['cobra']*100.00/test['rr_count']).round(2)
test

,exp_group,user_count,fe_count,rr_count,net_count,cobrm,cobra,ocara,fe2rr,g2n,fe2net,% cobrm,% cobra
0,CONTROL,196619,1174706,400892,247495,2280,48024,77226,34.13,61.74,21.07,0.57,11.98
1,TEST,171619,1296677,462976,269576,3071,64003,93808,35.70,58.23,20.79,0.66,13.82


In [17]:
df_group_tc = homogeneous_sample[['exp_group', 'user_id']].reset_index(drop=True)

## Funnel

In [18]:
city_str

"Bangalore', 'Jaipur', 'Hyderabad', 'Delhi"

In [19]:
fare_estimates_enriched = f"""
    
    SELECT
        user_id AS user_id,
        COUNT(DISTINCT fare_estimate_id) fe
    FROM
        pricing.fare_estimates_enriched fee
    WHERE 
        yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND city IN ('{city_str}')
            AND service_name IN ('{service_str}')
            AND lower(platform) IN ('android')
    GROUP BY 1
""" 

df_fare_estimates_enriched = pd.read_sql(fare_estimates_enriched, connection)
df_fare_estimates_enriched.to_csv(local_datasets + 'df_fare_estimates_enriched_v1.csv' , index=False)

In [20]:
order_logs_fact = f"""

    SELECT 
        customer_id,
        COUNT(DISTINCT order_id) gross_count,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != true THEN order_id END) net_count,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'expired' THEN order_id END) expired,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'aborted' THEN order_id END) aborted,

        approx_percentile(CASE WHEN modified_order_status = 'COBRM' THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END,0.50) as median_cobrm_time,
        approx_percentile(CASE WHEN modified_order_status = 'COBRA' THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END,0.50) as median_cobra_time,
        approx_percentile(CASE WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM')  THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END,0.50) as median_ttc
            
    FROM 
        orders.order_logs_fact fact
    WHERE 
        yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND city_name IN ('{city_str}')
            AND service_obj_service_name IN ('{service_str}')
            AND channel_host IN ('android')
    GROUP BY 1
""" 
df_order_logs_fact = pd.read_sql(order_logs_fact, connection)
df_order_logs_fact.to_csv(local_datasets + 'df_order_logs_fact_v1.csv' , index=False)

In [21]:
banner_monetisation = f"""

    SELECT 
        user_id,
        sub_type,
        COUNT(DISTINCT CASE WHEN sub_type = 'nativeBanner' AND action IN ('view') THEN unique_id END) AS views_nb,
        COUNT(DISTINCT CASE WHEN sub_type = 'nativeVideoBanner' AND action IN ('view') THEN unique_id END) AS views_nvb,
        COUNT(DISTINCT CASE WHEN sub_type = 'nativeBanner' AND action IN ('click') THEN unique_id END) AS clicks_nb ,
        COUNT(DISTINCT CASE WHEN sub_type = 'nativeVideoBanner' AND action IN ('click') THEN unique_id END) AS clicks_nvb 
    FROM
        (
        SELECT
            yyyymmdd,
            event_props_user_id user_id,
            event_props_action action,
            event_props_order_status order_status,
            event_props_sub_type sub_type,
            event_props_gam_banner_constant gam_banner_constant,
            event_props_user_id || '-' || event_props_action || '-' || CAST(epoch/1000 AS VARCHAR) unique_id
        FROM    
            clevertap.customer_banner_monetisation_immutable
        WHERE  
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND event_props_current_city IN ('{city_str}')
            AND event_props_action IN ('view', 'click')
            AND event_props_type = 'GAMBanner'
            AND event_props_sub_type LIKE 'native%'
            AND event_props_screen_name = 'CaptainSearchScreen'
            AND event_props_placement IN ( 'CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd', 'CaptainSearchScreenSlot:d1d2227b-92be-4318-b765-89405bcd1787')
            
        GROUP BY 1,2,3,4,5,6,7
        ) 
    GROUP BY 1,2
""" 
df_banner_monetisation = pd.read_sql(banner_monetisation, connection)
df_banner_monetisation.to_csv(local_datasets + 'df_banner_monetisation_v1.csv' , index=False)

In [22]:
df_fare_estimates_enriched = pd.read_csv(local_datasets + 'df_fare_estimates_enriched_v1.csv')
df_order_logs_fact = pd.read_csv(local_datasets + 'df_order_logs_fact_v1.csv')
df_banner_monetisation = pd.read_csv(local_datasets + 'df_banner_monetisation_v1.csv')

In [23]:
df_group_tc.head(2)

,exp_group,user_id
0,CONTROL,62621cd9e129df17a87ba2eb
1,CONTROL,62d024f3f316915cd207f889


In [24]:
df_fare_estimates_enriched.head(2)

,user_id,fe
0,63ef1e3dec8d7b26c2e3b6ce,14
1,5fcdf3035441f8183545dc6a,3


In [25]:
merge_df1 = pd.merge(df_group_tc,
                     df_fare_estimates_enriched,
                     how='left',
                     on = ['user_id']
                    )
merge_df1.head(2)

,exp_group,user_id,fe
0,CONTROL,62621cd9e129df17a87ba2eb,8
1,CONTROL,62d024f3f316915cd207f889,8


In [26]:
df_order_logs_fact.head(2)

,customer_id,gross_count,net_count,cobrm,cobra,ocara,expired,aborted,median_cobrm_time,median_cobra_time,median_ttc
0,59c924fbcd75ba844e44dcd5,1,1,0,0,0,0,0,NaN,NaN,NaN
1,627d06a40acf8597f889fe13,12,11,0,0,1,0,0,NaN,NaN,286.0


In [27]:
merge_df2 = pd.merge(merge_df1,
                     df_order_logs_fact,
                     how='left',
                     left_on=['user_id'],
                     right_on=['customer_id']
                    )
merge_df2.head(2)

,exp_group,user_id,fe,customer_id,gross_count,net_count,cobrm,cobra,ocara,expired,aborted,median_cobrm_time,median_cobra_time,median_ttc
0,CONTROL,62621cd9e129df17a87ba2eb,8,62621cd9e129df17a87ba2eb,5,3,0,0,2,0,0,NaN,NaN,410.0
1,CONTROL,62d024f3f316915cd207f889,8,62d024f3f316915cd207f889,4,2,0,1,1,0,0,NaN,242.0,245.0


In [28]:
df_banner_monetisation.head(2)

,user_id,sub_type,views_nb,views_nvb,clicks_nb,clicks_nvb
0,5cceff3220e484444770b358,nativeBanner,3,0,0,0
1,658c17aebd47a3ff835c6102,nativeBanner,4,0,0,0


In [29]:
merge_df3 = pd.merge(merge_df2,
                     df_banner_monetisation,
                     how='left',
                     left_on=['user_id'],
                     right_on=['user_id']
                    )
merge_df3.head(2)

,exp_group,user_id,fe,customer_id,gross_count,net_count,cobrm,cobra,ocara,expired,aborted,median_cobrm_time,median_cobra_time,median_ttc,sub_type,views_nb,views_nvb,clicks_nb,clicks_nvb
0,CONTROL,62621cd9e129df17a87ba2eb,8,62621cd9e129df17a87ba2eb,5,3,0,0,2,0,0,NaN,NaN,410.0,NaN,NaN,NaN,NaN,NaN
1,CONTROL,62d024f3f316915cd207f889,8,62d024f3f316915cd207f889,4,2,0,1,1,0,0,NaN,242.0,245.0,NaN,NaN,NaN,NaN,NaN


In [30]:
merge_df3['sub_type'] = merge_df3['sub_type'].fillna('No-Ads')

In [31]:
df_ana1 = merge_df3.groupby(['exp_group'])\
.agg(customers=('customer_id', 'nunique'),
     fe_count=('fe', 'sum'),
     gross_count=('gross_count', 'sum'),
     net_count=('net_count', 'sum'),
     cobrm=('cobrm', 'sum'),
     cobra=('cobra', 'sum'),
     median_cobrm_time=('median_cobrm_time', 'median'),
     median_cobra_time=('median_cobra_time', 'median'),
     median_ttc=('median_ttc', 'median'),
     ocara=('ocara', 'sum'),
     expired=('expired', 'sum'),
     aborted=('aborted', 'sum'),
     views_nb=('views_nb', 'sum'),
     clicks_nb=('clicks_nb', 'sum'),
     views_nvb=('views_nvb', 'sum'),
     clicks_nvb=('clicks_nvb', 'sum'),
     
).reset_index()

In [32]:
df_ana1

,exp_group,customers,fe_count,gross_count,net_count,cobrm,cobra,median_cobrm_time,median_cobra_time,median_ttc,ocara,expired,aborted,views_nb,clicks_nb,views_nvb,clicks_nvb
0,CONTROL,196619,1174706,407797,250072,2338,49084,142.0,147.0,260.0,78686,24794,1140,0.0,0.0,0.0,0.0
1,TEST,171619,1356151,497309,284899,3446,69985,145.0,151.0,257.0,101258,34044,1362,255433.0,1676.0,23492.0,193.0


In [52]:
df_ana1.to_clipboard(index=False)

In [34]:
df_ans2 = merge_df3.groupby(['exp_group', 'sub_type'])\
.agg(customers=('customer_id', 'nunique'),
     fe_count=('fe', 'sum'),
     gross_count=('gross_count', 'sum'),
     net_count=('net_count', 'sum'),
     cobrm=('cobrm', 'sum'),
     cobra=('cobra', 'sum'),
     median_cobrm_time=('median_cobrm_time', 'median'),
     median_cobra_time=('median_cobra_time', 'median'),
     median_ttc=('median_ttc', 'median'),
     ocara=('ocara', 'sum'),
     expired=('expired', 'sum'),
     aborted=('aborted', 'sum'),
     views_nb=('views_nb', 'sum'),
     clicks_nb=('clicks_nb', 'sum'),
     views_nvb=('views_nvb', 'sum'),
     clicks_nvb=('clicks_nvb', 'sum'),
     
).reset_index()

In [53]:
df_ans2

,exp_group,sub_type,customers,fe_count,gross_count,net_count,cobrm,cobra,median_cobrm_time,median_cobra_time,median_ttc,ocara,expired,aborted,views_nb,clicks_nb,views_nvb,clicks_nvb
0,CONTROL,No-Ads,196619,1174706,407797,250072,2338,49084,142.0,147.0,260.0,78686,24794,1140,0.0,0.0,0.0,0.0
1,TEST,nativeBanner,158735,1200999,439434,253498,2956,61163,144.0,151.0,257.0,89268,29333,1197,255433.0,1676.0,0.0,0.0
2,TEST,nativeVideoBanner,19484,155152,57875,31401,490,8822,151.0,157.0,264.0,11990,4711,165,0.0,0.0,23492.0,193.0


In [54]:
df_ans2.to_clipboard(index=False)

In [37]:
merge_df3

,exp_group,user_id,fe,customer_id,gross_count,net_count,cobrm,cobra,ocara,expired,aborted,median_cobrm_time,median_cobra_time,median_ttc,sub_type,views_nb,views_nvb,clicks_nb,clicks_nvb
0,CONTROL,62621cd9e129df17a87ba2eb,8,62621cd9e129df17a87ba2eb,5,3,0,0,2,0,0,NaN,NaN,410.0,No-Ads,NaN,NaN,NaN,NaN
1,CONTROL,62d024f3f316915cd207f889,8,62d024f3f316915cd207f889,4,2,0,1,1,0,0,NaN,242.0,245.0,No-Ads,NaN,NaN,NaN,NaN
2,CONTROL,6479b403b9346af6ad8e0589,21,6479b403b9346af6ad8e0589,5,4,0,1,0,0,0,NaN,181.0,181.0,No-Ads,NaN,NaN,NaN,NaN
3,CONTROL,65e445a86e7643d2019e3f6d,3,65e445a86e7643d2019e3f6d,1,0,0,1,0,0,0,NaN,191.0,191.0,No-Ads,NaN,NaN,NaN,NaN
4,CONTROL,65348300b5145a649415696d,21,65348300b5145a649415696d,4,1,0,0,3,0,0,NaN,NaN,860.0,No-Ads,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374833,TEST,65eadd7c0491982e0fc8a99e,3,65eadd7c0491982e0fc8a99e,1,1,0,0,0,0,0,NaN,NaN,NaN,nativeBanner,1.0,0.0,0.0,0.0
374834,TEST,5a22b58cc50f5c655a06cf37,10,5a22b58cc50f5c655a06cf37,4,1,0,0,3,0,0,NaN,NaN,101.0,nativeBanner,3.0,0.0,0.0,0.0
374835,TEST,617aad5469ebf832f9e24eec,4,617aad5469ebf832f9e24eec,2,2,0,0,0,0,0,NaN,NaN,NaN,nativeBanner,1.0,0.0,0.0,0.0
374836,TEST,5dc2d2193087fd49363c1d1d,8,5dc2d2193087fd49363c1d1d,3,1,0,2,0,0,0,NaN,159.0,159.0,nativeBanner,1.0,0.0,0.0,0.0


In [45]:
grouper_column = merge_df3[['exp_group', 'customer_id', 'sub_type']][~merge_df3['customer_id'].isna()]
grouper_column

,exp_group,customer_id,sub_type
0,CONTROL,62621cd9e129df17a87ba2eb,No-Ads
1,CONTROL,62d024f3f316915cd207f889,No-Ads
2,CONTROL,6479b403b9346af6ad8e0589,No-Ads
3,CONTROL,65e445a86e7643d2019e3f6d,No-Ads
4,CONTROL,65348300b5145a649415696d,No-Ads
...,...,...,...
374833,TEST,65eadd7c0491982e0fc8a99e,nativeBanner
374834,TEST,5a22b58cc50f5c655a06cf37,nativeBanner
374835,TEST,617aad5469ebf832f9e24eec,nativeBanner
374836,TEST,5dc2d2193087fd49363c1d1d,nativeBanner


## pre-post

In [39]:
start_date_new = '20240225'
end_date_new = '20240229'

In [40]:
city_str

"Bangalore', 'Jaipur', 'Hyderabad', 'Delhi"

In [41]:
service_str

"CabEconomy', 'Bike Lite', 'Link', 'Auto"

In [42]:
user_base_query = f"""

    WITH fe_tbl AS (

        SELECT
            yyyymmdd,
            user_id as user_id,
            fare_estimate_id
        FROM
            hive.pricing.fare_estimates_enriched
        WHERE 
            yyyymmdd >= '{start_date_new}'
            AND yyyymmdd <= '{end_date_new}'
            AND city IN ('{city_str}')
            AND service_name IN ('{service_str}')
            AND lower(platform) IN ('android')
        ),

        rr_net_tbl as (

        SELECT   
            yyyymmdd,
            city_name AS city,
            service_obj_service_name AS service_name,
            customer_id,
            estimate_id AS fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag,
            modified_order_status
        FROM orders.order_logs_fact ols
        WHERE
            yyyymmdd >= '{start_date_new}'
            AND yyyymmdd <= '{end_date_new}'
            AND city_name IN ('{city_str}')
            AND service_obj_service_name IN ('{service_str}')
            AND channel_host IN ('android')
        )

        SELECT
            case 
            when yyyymmdd >= '20240301' then 'post'
            when yyyymmdd <= '20240229' then 'pre'
            end period,
            user_id,
            COUNT(distinct fare_estimate_id) AS fe_count,
            COUNT(distinct order_id) AS rr_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count,

            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRM' THEN order_id END) cobrm,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'COBRA' THEN order_id END) cobra,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'OCARA' THEN order_id END) ocara,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'expired' THEN order_id END) expired,
            COUNT(DISTINCT CASE WHEN modified_order_status = 'aborted' THEN order_id END) aborted,

            COALESCE(TRY(COUNT(distinct order_id)*100.00/COUNT(distinct fare_estimate_id)), 0) FE_RR,
            COALESCE(TRY(COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END)*100.00/COUNT(distinct order_id)), 0) G2N,
            COALESCE(TRY(COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END)*100.00/COUNT(distinct fare_estimate_id)), 0) FE_NET

        FROM        
            (
            SELECT
                fe_tbl.*,
                rr_net_tbl.customer_id,
                rr_net_tbl.order_id,
                rr_net_tbl.order_status,
                rr_net_tbl.modified_order_status,
                rr_net_tbl.spd_fraud_flag
            FROM
                fe_tbl
            LEFT JOIN
                rr_net_tbl
                ON fe_tbl.yyyymmdd = rr_net_tbl.yyyymmdd
                AND fe_tbl.fare_estimate_id = rr_net_tbl.fare_estimate_id
            )
        group by 1,2
                    
"""

df_user_base = pd.read_sql(user_base_query, connection)
df_user_base.to_csv(local_datasets + 'df_user_base_new_prepost.csv' , index=False)

In [43]:
df_user_base = pd.read_csv(local_datasets + 'df_user_base_new_prepost.csv')
df_user_base

,period,user_id,fe_count,rr_count,net_count,cobrm,cobra,ocara,expired,aborted,FE_RR,G2N,FE_NET
0,pre,60fd2550fe2f60544e876c62,15,0,0,0,0,0,0,0,0.00,0.0,0.00
1,pre,5c6e624d35478b560277b623,2,1,1,0,0,0,0,0,50.00,100.0,50.00
2,pre,5e60a6bba9f9b241e60ab2ec,2,0,0,0,0,0,0,0,0.00,0.0,0.00
3,pre,64f6bb7b8d4f195d710463ca,5,2,2,0,0,0,0,0,40.00,100.0,40.00
4,pre,627e657ac75b90f94c09ac62,3,1,1,0,0,0,0,0,33.33,100.0,33.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3055978,pre,65dab756e71a30df9619e492,4,1,1,0,0,0,0,0,25.00,100.0,25.00
3055979,pre,5c4040344a267149c7721c3f,2,0,0,0,0,0,0,0,0.00,0.0,0.00
3055980,pre,63022ed6db8f96bafaa761db,4,2,1,0,0,1,0,0,50.00,50.0,25.00
3055981,pre,65883f5d80d337b95aa24f29,1,0,0,0,0,0,0,0,0.00,0.0,0.00


In [46]:
grouper_column.head(3)

,exp_group,customer_id,sub_type
0,CONTROL,62621cd9e129df17a87ba2eb,No-Ads
1,CONTROL,62d024f3f316915cd207f889,No-Ads
2,CONTROL,6479b403b9346af6ad8e0589,No-Ads


In [47]:
df_compare = pd.merge(grouper_column, 
                      df_user_base,
                      how='inner',
                      left_on = ['customer_id'],
                      right_on = ['user_id']
                     )
df_compare.head(5)

,exp_group,customer_id,sub_type,period,user_id,fe_count,rr_count,net_count,cobrm,cobra,ocara,expired,aborted,FE_RR,G2N,FE_NET
0,CONTROL,62621cd9e129df17a87ba2eb,No-Ads,pre,62621cd9e129df17a87ba2eb,2,0,0,0,0,0,0,0,0.00,0.0,0.00
1,CONTROL,62d024f3f316915cd207f889,No-Ads,pre,62d024f3f316915cd207f889,12,4,1,0,0,1,1,1,33.33,25.0,8.33
2,CONTROL,64d83d525e363621973816b1,No-Ads,pre,64d83d525e363621973816b1,9,2,1,0,0,1,0,0,22.22,50.0,11.11
3,CONTROL,63f551bef1e050b2fa96d982,No-Ads,pre,63f551bef1e050b2fa96d982,4,2,2,0,0,0,0,0,50.00,100.0,50.00
4,CONTROL,613b0ebab6866cce867e6eff,No-Ads,pre,613b0ebab6866cce867e6eff,9,1,0,0,0,1,0,0,11.11,0.0,0.00


In [48]:
df_compare = df_compare[(df_compare['exp_group'] == 'CONTROL') | (df_compare['sub_type'] != 'No-Ads')]
df_compare

,exp_group,customer_id,sub_type,period,user_id,fe_count,rr_count,net_count,cobrm,cobra,ocara,expired,aborted,FE_RR,G2N,FE_NET
0,CONTROL,62621cd9e129df17a87ba2eb,No-Ads,pre,62621cd9e129df17a87ba2eb,2,0,0,0,0,0,0,0,0.00,0.0,0.00
1,CONTROL,62d024f3f316915cd207f889,No-Ads,pre,62d024f3f316915cd207f889,12,4,1,0,0,1,1,1,33.33,25.0,8.33
2,CONTROL,64d83d525e363621973816b1,No-Ads,pre,64d83d525e363621973816b1,9,2,1,0,0,1,0,0,22.22,50.0,11.11
3,CONTROL,63f551bef1e050b2fa96d982,No-Ads,pre,63f551bef1e050b2fa96d982,4,2,2,0,0,0,0,0,50.00,100.0,50.00
4,CONTROL,613b0ebab6866cce867e6eff,No-Ads,pre,613b0ebab6866cce867e6eff,9,1,0,0,0,1,0,0,11.11,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129237,TEST,65c490724b75c14b5e999cbb,nativeBanner,pre,65c490724b75c14b5e999cbb,1,0,0,0,0,0,0,0,0.00,0.0,0.00
129238,TEST,62cfd722e9c333c3c3a75a6e,nativeBanner,pre,62cfd722e9c333c3c3a75a6e,2,1,1,0,0,0,0,0,50.00,100.0,50.00
129239,TEST,60001aed87d741a2fa249b7c,nativeBanner,pre,60001aed87d741a2fa249b7c,1,0,0,0,0,0,0,0,0.00,0.0,0.00
129240,TEST,62f3eb8142eb0a4371d7be88,nativeVideoBanner,pre,62f3eb8142eb0a4371d7be88,10,2,2,0,0,0,0,0,20.00,100.0,20.00


In [49]:
dff = df_compare.groupby(['period', 'exp_group'])\
.agg(customers=('customer_id', 'nunique'),
     fe_count=('fe_count', 'sum'),
     gross_count=('rr_count', 'sum'),
     net_count=('net_count', 'sum'),
     cobrm=('cobrm', 'sum'),
     cobra=('cobra', 'sum'),
     ocara=('ocara', 'sum'),
     expired=('expired', 'sum'),
     aborted=('aborted', 'sum')
).reset_index()
dff

,period,exp_group,customers,fe_count,gross_count,net_count,cobrm,cobra,ocara,expired,aborted
0,pre,CONTROL,62821,285566,100344,61040,562,13712,19178,5574,199
1,pre,TEST,63929,331430,114332,68146,640,15986,21990,7265,216


In [50]:
dff2 = df_compare.groupby(['period', 'exp_group','sub_type'])\
.agg(customers=('customer_id', 'nunique'),
     fe_count=('fe_count', 'sum'),
     gross_count=('rr_count', 'sum'),
     net_count=('net_count', 'sum'),
     cobrm=('cobrm', 'sum'),
     cobra=('cobra', 'sum'),
     ocara=('ocara', 'sum'),
     expired=('expired', 'sum'),
     aborted=('aborted', 'sum')
).reset_index()
dff2

,period,exp_group,sub_type,customers,fe_count,gross_count,net_count,cobrm,cobra,ocara,expired,aborted
0,pre,CONTROL,No-Ads,62821,285566,100344,61040,562,13712,19178,5574,199
1,pre,TEST,nativeBanner,59525,297383,102687,61204,560,14376,19703,6570,197
2,pre,TEST,nativeVideoBanner,6896,34047,11645,6942,80,1610,2287,695,19


In [51]:
59525+34047

93572